# Lab: Swiss-Prot enzyme dataset — sequence, structure, and active-site similarity

**Goal:** Design a **train / validation / test split** for a machine-learning model studying enzyme active sites. Start with **reviewed Swiss-Prot enzymes**, then compare similarity at three levels: **MMseqs2 (sequence)**, **Foldseek (global structure)**, and **Folddisco (local 3D motifs)**.

**Your job is to design the dataset, not just run the tools.** Some cells are starter code; exercises deliberately remain open-ended.

**Colab:** Runtime → Run all (initial demonstration), then modify thresholds and add your own cells. CPU is sufficient for the small example.

⚠️ **Storage:** Google Colab's `/content` filesystem is temporary. A complete Swiss-Prot enzyme structure collection may require **many GB or more**, especially with intermediate databases and indexes. Start with 100–300 sequences and 10–30 structures; monitor disk space, and use a machine with substantial storage for scaling. Saving files to Drive does not increase Colab's local disk quota.

## 0 · Setup
The installers below target Linux/x86-64 Colab. MMseqs2 is installed via apt; Foldseek and Folddisco via their official binaries. The downloads and commands may change upstream, so consult the linked READMEs if a download fails.

In [ ]:
%pip -q install pandas requests biopython matplotlib
!apt-get -qq update && apt-get -qq install -y mmseqs2 mafft > /dev/null
!wget -q https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz -O /tmp/foldseek.tar.gz
!tar -xzf /tmp/foldseek.tar.gz -C /content
!wget -q https://mmseqs.com/folddisco/folddisco-linux-x86_64.tar.gz -O /tmp/folddisco.tar.gz
!tar -xzf /tmp/folddisco.tar.gz -C /content
import os, shutil
os.environ['PATH']='/content/foldseek/bin:/content/folddisco/bin:'+os.environ['PATH']
for tool in ['mmseqs','foldseek','folddisco','mafft']:
    print(tool, '→', shutil.which(tool))
!df -h /content | tail -1

## 1 · Retrieve only reviewed Swiss-Prot enzymes

UniProtKB contains reviewed (**Swiss-Prot**) and unreviewed (**TrEMBL**) sequences. We require `reviewed:true` and an EC number (`ec:*`) as a practical initial enzyme definition. **An EC annotation is not proof that catalytic residues are annotated.**

This first API page is a **convenience sample, not a random or representative sample**. For a serious dataset, paginate the UniProt API and document selection criteria, release date, and missing annotations.

In [ ]:
from pathlib import Path
import io, requests, pandas as pd
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
WORK = Path('/content/enzyme_lab'); WORK.mkdir(exist_ok=True)
N_PROTEINS = 200  # try 100–300 first
query = '(reviewed:true) AND (ec:*)'
url = 'https://rest.uniprot.org/uniprotkb/search'
params = {'query':query, 'format':'tsv', 'fields':'accession,id,protein_name,ec,sequence,ft_act_site', 'size':N_PROTEINS}
response = requests.get(url, params=params, timeout=120); response.raise_for_status()
proteins = pd.read_csv(io.StringIO(response.text), sep='\t', dtype=str).fillna('')
proteins = proteins.rename(columns={'Entry':'accession','Sequence':'sequence'})
assert len(proteins) > 0 and {'accession','sequence'}.issubset(proteins.columns)
proteins['length'] = proteins.sequence.str.len()
proteins.to_csv(WORK/'metadata.csv', index=False)
SeqIO.write([SeqRecord(Seq(s), id=a, description='') for a,s in zip(proteins.accession, proteins.sequence)], WORK/'enzymes.fasta', 'fasta')
print(f'{len(proteins)} reviewed enzymes; total residues {proteins.length.sum():,}')
display(proteins.head(5))

**Investigate:** What other definition of “enzyme” could you use? Which EC classes are represented? Is the API's first page biased? How would you obtain **all** matching Swiss-Prot proteins using the API `Link: ... rel="next"` pagination header? See [UniProt REST documentation](https://www.uniprot.org/help/api_queries).

Remember: for active-site supervision you may need experimental evidence / UniProt **ACT_SITE** annotations and coordinate mapping. Missing annotation is **unknown**, not a negative example.

## 2 · MMseqs2: cluster proteins by sequence

`easy-cluster FASTA PREFIX TMP` produces `PREFIX_cluster.tsv` (representative → member). Sequence identity and coverage affect cluster membership. The threshold below is **illustrative, not the answer**.

In [ ]:
import subprocess
seq_prefix = WORK/'seq_50'
cmd = ['mmseqs','easy-cluster',str(WORK/'enzymes.fasta'),str(seq_prefix),str(WORK/'tmp_seq'), '--min-seq-id','0.5','-c','0.8','--cov-mode','0','--threads','2']
subprocess.run(cmd, check=True)
seq_edges = pd.read_csv(str(seq_prefix)+'_cluster.tsv', sep='\t', header=None, names=['representative','member'])
seq_groups = seq_edges.rename(columns={'representative':'seq_cluster','member':'accession'})[['accession','seq_cluster']]
print('Sequence clusters:',seq_groups.seq_cluster.nunique())
display(seq_edges.head())

**Try:** Compare 30%, 50%, and 90% sequence identity. What happens to cluster sizes? Does keeping a cluster intact across splits stop all forms of information leakage?

## 3 · Retrieve predicted 3D structures (limited subset)

Use AlphaFold Protein Structure Database's API to look up structure files by UniProt accession. Only a **small number** are fetched by default. Some sequences may have no matching prediction; not all predicted regions are reliable. Distinguish **predicted structures** from experimental PDB structures.

**Storage warning:** Scale up only after estimating average file size, available disk, and temporary Foldseek/Folddisco indexing space.

In [ ]:
import time
STRUCTURE_LIMIT = 20
structure_dir = WORK/'structures'; structure_dir.mkdir(exist_ok=True)
structure_rows=[]
for accession in proteins.accession.head(STRUCTURE_LIMIT):
    try:
        r=requests.get(f'https://alphafold.ebi.ac.uk/api/prediction/{accession}',timeout=25)
        if r.status_code != 200: continue
        entries=r.json()
        if not entries: continue
        pdb_url=entries[0].get('pdbUrl')
        if not pdb_url: continue
        dst=structure_dir/f'{accession}.pdb'
        if not dst.exists():
            download=requests.get(pdb_url,timeout=45); download.raise_for_status()
            dst.write_bytes(download.content)
        structure_rows.append({'accession':accession,'structure_path':str(dst),'source':'AlphaFoldDB'})
    except (requests.RequestException, ValueError) as exc:
        print('Skipped', accession, str(exc)[:80])
    time.sleep(0.1)
structures=pd.DataFrame(structure_rows,columns=['accession','structure_path','source'])
structures.to_csv(WORK/'structures_manifest.csv',index=False)
print('Structures downloaded:',len(structures))
print('PDB files MB:',round(sum(p.stat().st_size for p in structure_dir.glob('*.pdb'))/1e6,2))
!df -h /content | tail -1
display(structures.head())

**Think:** Download all structures *before* clustering, or first reduce redundancy by sequence and then fetch representative structures? How do you account for missing predictions, domains, low-confidence regions, and proteins with multiple chains?

## 4 · Foldseek: cluster by **global structural** similarity

Foldseek encodes structural neighborhoods using 3Di states. `easy-cluster` accepts a directory of structures and outputs another representative→member TSV. Filenames/chain identifiers may differ from UniProt accessions, so **inspect identifiers instead of assuming an exact match**.

In [ ]:
if len(structures) >= 2:
    fold_prefix = WORK/'fold_cluster'
    subprocess.run(['foldseek','easy-cluster',str(structure_dir),str(fold_prefix),str(WORK/'tmp_fold'),'--threads','2'],check=True)
    fold_edges=pd.read_csv(str(fold_prefix)+'_clu.tsv',sep='\t',header=None,names=['representative','member'])
    print('Foldseek clusters:',fold_edges.representative.nunique())
    display(fold_edges.head(10))
else:
    print('Not enough structures for clustering. Try different Swiss-Prot accessions.')

**Try:** How much do sequence and structure clusters agree? Does a shared global fold guarantee a shared enzyme function or active-site chemistry? What Foldseek thresholds might be relevant? Investigate `foldseek easy-cluster -h`.

## 5 · Folddisco: search a **local 3D active-site motif**

Unlike MMseqs2 and Foldseek `easy-cluster`, Folddisco's main workflow searches an **indexed structure set** for geometric arrangements of **specified residues**. The students must devise how to transform motif matches into active-site groups, and must carefully handle non-hits and residue-coordinate mappings.

Build a small index first; then select a protein with reliable catalytic-site residue positions. **Do not assume UniProt feature sequence positions always equal structure PDB residue numbers.** Check the PDB chain and numbering first.

Example CLI form (choose a real motif yourself):

```bash
folddisco index -p /content/enzyme_lab/structures -i /content/enzyme_lab/motif_index -t 2
folddisco query -i /content/enzyme_lab/motif_index -p /path/to/query.pdb -q A57,A102,A195 -t 2
```

`A57,A102,A195` is merely **syntax** (chain A, positions 57/102/195), **not verified catalytic residues for your query structure**. Folddisco also supports batch queries using a TSV of structure paths and motif residues.

In [ ]:
if len(structures) >= 2:
    subprocess.run(['folddisco','index','-p',str(structure_dir),'-i',str(WORK/'motif_index'),'-t','2'],check=True)
    print('Motif index built. Inspect UniProt ACT_SITE annotations and your PDB numbering before querying.')
else:
    print('Download at least two structures first.')

In [ ]:
# TODO (student): Choose a protein with >= 3 known catalytic or metal-binding residues.
# Verify PDB chain IDs and numbering, then uncomment and edit:
# query_pdb = structure_dir / 'YOUR_ACCESSION.pdb'
# motif_residues = 'A10,A50,A90'  # REPLACE with verified residues
# result = subprocess.run(['folddisco','query','-i',str(WORK/'motif_index'),
#                          '-p',str(query_pdb),'-q',motif_residues,'-t','2'],
#                         capture_output=True,text=True,check=True)
# print(result.stdout[:3000])

**Challenge:** How would you make active-site clusters from multiple motif queries and match scores? Must all enzymes possess an annotated catalytic motif? How do you avoid connecting unrelated enzymes through weak or generic motifs? Is your graph clustering transitive? How would you detect similar catalytic sites in distinct folds?

## 6 · Optional: learn how to obtain an MSA

An **MSA** aligns homologous protein sequences. This demonstration runs a *shallow alignment* of nearby sequences from the toy dataset; it is **not equivalent to a high-coverage evolutionary MSA**. For high-quality per-protein MSAs you must search a much larger homolog database (e.g., UniRef/UniProt using MMseqs2 or jackhmmer) and align/filter hits. Downloading UniRef locally can be far too large for free Colab; consider remote services or a storage-equipped machine, subject to server usage policies.

Run the example below to practice the mechanics of `mmseqs easy-search` and `mafft`.

In [ ]:
from Bio import AlignIO
query_accession=proteins.accession.iloc[0]
query_rec=next(x for x in SeqIO.parse(WORK/'enzymes.fasta','fasta') if x.id==query_accession)
SeqIO.write([query_rec],WORK/'query.fasta','fasta')
subprocess.run(['mmseqs','easy-search',str(WORK/'query.fasta'),str(WORK/'enzymes.fasta'),str(WORK/'hits.m8'),str(WORK/'tmp_search'),'--threads','2'],check=True)
hits=pd.read_csv(WORK/'hits.m8',sep='\t',header=None)
related=set(hits.iloc[:,1].head(15).astype(str)) | {query_accession}
records=[rec for rec in SeqIO.parse(WORK/'enzymes.fasta','fasta') if rec.id in related]
print('Candidate sequences for tutorial alignment:',len(records))
if len(records)>=2:
    SeqIO.write(records,WORK/'msa_input.fasta','fasta')
    with open(WORK/'example_msa.fasta','w') as output:
        subprocess.run(['mafft','--auto',str(WORK/'msa_input.fasta')],stdout=output,stderr=subprocess.DEVNULL,check=True)
    print('Saved example_msa.fasta; first alignment entries:')
    print((WORK/'example_msa.fasta').read_text()[:800])
else:
    print('Too few homologs in this small sample. Expand the reference sequence database.')

## 7 · YOUR TASK: Design leakage-resistant train / validation / test splits

Create a manifest with at least these columns:

| accession | EC label | sequence cluster | structure cluster | active-site motif group | split | structure source | notes |
|---|---|---|---|---|---|---|---|

You can add missingness flags, structural confidence, annotation evidence, or group IDs. **Do not silently label proteins without catalytic annotations as nonenzymes or negative sites.**

Discuss: (1) What belongs together in one split? (2) Are sequence/global-fold/motif groupings best used individually or combined? (3) How will you handle chains, isoforms, homologs, families, and related motifs? (4) Could shared active-site motifs leak labels between partitions? (5) Which held-out generalization claim can you honestly make?

**Starter scaffold — intentionally leaves all split assignments for you:**

In [ ]:
manifest = proteins[['accession']].copy()
manifest['ec_label'] = proteins['EC number'] if 'EC number' in proteins.columns else ''
manifest = manifest.merge(seq_groups, on='accession',how='left')
manifest['structure_cluster'] = pd.NA     # TODO: map Foldseek identifiers to accessions
manifest['active_site_motif_group'] = pd.NA # TODO: use Folddisco motif results
manifest['split'] = pd.NA                # TODO: assign after grouping (not row-by-row randomly)
manifest['structure_source'] = manifest.accession.map(dict(zip(structures.accession,structures.source)))
manifest['notes'] = ''
manifest.to_csv(WORK/'starter_manifest.csv',index=False)
display(manifest.head())
print('Saved:',WORK/'starter_manifest.csv')

### Submission (short)
1. `dataset_manifest.csv` with train/validation/test assignments and missingness clearly marked.
2. A one-page explanation of how you sourced **only reviewed Swiss-Prot enzymes** and handled structural/active-site evidence.
3. Sequence, global-structure, and local-motif clustering methods (including parameters) and leakage checks.
4. A paragraph describing what could still leak, and how you would scale the workflow.

**Documentation:** [UniProt REST](https://www.uniprot.org/help/api_queries) · [MMseqs2](https://github.com/soedinglab/MMseqs2) · [Foldseek](https://github.com/steineggerlab/foldseek) · [Folddisco](https://github.com/steineggerlab/folddisco) · [AlphaFold DB](https://alphafold.ebi.ac.uk/)

**Note:** This is a teaching scaffold, not a validated production pipeline; API response formats, binary availability, and biological residue mapping should be verified for your actual dataset.